# Exercițiu Laborator 6 — Inferență Bayesiană (Modelul Conjugat Gamma–Poisson)

## Enunțul problemei
O companie de telecomunicații dorește să estimeze rata medie $\lambda > 0$ de apeluri primite de un call center într-o oră.
* **Date observate:** În $n = 10$ ore s-au primit un total de $\sum_{i=1}^{10} k_i = 180$ apeluri (rata medie observată fiind $\bar{k} = 18$ apeluri/oră).
* **Distribuția verosimilității:** Numărul de apeluri dintr-o oră urmează o distribuție Poisson cu parametrul $\lambda$:
  $$P(X = k \mid \lambda) = e^{-\lambda} \frac{\lambda^k}{k!}.$$
* **Distribuția a priori:** $\lambda \sim \text{Gamma}(\alpha_0, \beta_0)$ (parametrizarea rată $\beta_0 > 0$), având densitatea:
  $$p(\lambda) = \frac{\beta_0^{\alpha_0}}{\Gamma(\alpha_0)} \lambda^{\alpha_0 - 1} e^{-\beta_0 \lambda},$$
  cu media $\mathbb{E}[\lambda] = \frac{\alpha_0}{\beta_0}$, dispersia $\text{Var}(\lambda) = \frac{\alpha_0}{\beta_0^2}$ și modul $\text{Mod}(\lambda) = \frac{\max(\alpha_0 - 1, 0)}{\beta_0}$.

**Cerințe:**
* Alegerea argumentată a parametrilor a priori $\alpha_0$ și $\beta_0$.
* Determinarea distribuției a posteriori pentru $\lambda$.
* Calculul intervalului **HDI de 94%** și al **celei mai probabile valori a lui $\lambda$** (modul a posteriori), inclusiv vizualizarea cu `arviz.plot_posterior`.

## 1. Derivarea analitică a distribuției a posteriori (Conjugarea Gamma–Poisson)

Pentru cele $n = 10$ observații independente $k_1, \dots, k_n$ cu suma $S = \sum_{i=1}^n k_i = 180$, funcția de verosimilitate este:
$$
L(\lambda) = \prod_{i=1}^n e^{-\lambda} \frac{\lambda^{k_i}}{k_i!} \propto \lambda^{\sum_{i=1}^n k_i} e^{-n\lambda} = \lambda^{180} e^{-10\lambda}.
$$

Aplicând **Teorema lui Bayes**, densitatea a posteriori este proporțională cu produsul dintre prior și verosimilitate:
$$
p(\lambda \mid k_1, \dots, k_n) \propto p(\lambda) \cdot L(\lambda) \propto \left(\lambda^{\alpha_0 - 1} e^{-\beta_0 \lambda}\right) \cdot \left(\lambda^{180} e^{-10\lambda}\right) = \lambda^{(\alpha_0 + 180) - 1} e^{-(\beta_0 + 10)\lambda}.
$$

Recunoaștem nucleul unei distribuții **Gamma** cu parametrii actualizați:
$$
\lambda \mid \text{date} \sim \text{Gamma}\left(\alpha_{\text{post}} = \alpha_0 + \sum_{i=1}^n k_i, \; \beta_{\text{post}} = \beta_0 + n\right) = \text{Gamma}(\alpha_0 + 180, \; \beta_0 + 10).
$$

### Cum alegem parametrii a priori $\alpha_0$ și $\beta_0$?
Parametrii $\alpha_0$ și $\beta_0$ au o interpretare directă ca **pseudo-observații istorice**: $\alpha_0$ reprezintă numărul total de apeluri observate anterior într-un interval echivalent de $\beta_0$ ore:
1. **Alegerea unui prior slab informativ (neinformativ):** Dacă nu dorim ca priorul să influențeze semnificativ datele curente (în special deoarece compania menționează că rata variază în funcție de perioada zilei), alegem parametri foarte mici, de exemplu $\alpha_0 = 1, \beta_0 = 0.01$ (o distribuție Exponențială foarte plată cu media $100$ și deviația standard $100$) sau $\alpha_0 = 0.5, \beta_0 = 0.001$.
   * Pentru $\alpha_0 = 1, \beta_0 = 0.01$:
     $$\lambda \mid \text{date} \sim \text{Gamma}(181, \; 10.01).$$
   * **Cea mai probabilă valoare (modul a posteriori):**
     $$\lambda_{\text{MAP}} = \frac{\alpha_{\text{post}} - 1}{\beta_{\text{post}}} = \frac{181 - 1}{10.01} = \frac{180}{10.01} \approx 17.982 \text{ apeluri/oră}.$$
   * **Media a posteriori:** $\mathbb{E}[\lambda \mid \text{date}] = \frac{181}{10.01} \approx 18.082$ apeluri/oră.
   * **Intervalul HDI de 94%:** $[15.58, \; 20.62]$ apeluri/oră.

2. **Alegerea unui prior informativ din date istorice:** Dacă din istoricul call center-ului cunoaștem o rată medie $\mu_0$ și o deviație standard $\sigma_0$, rezolvăm sistemul $\frac{\alpha_0}{\beta_0} = \mu_0$ și $\frac{\alpha_0}{\beta_0^2} = \sigma_0^2$, de unde:
   $$\beta_0 = \frac{\mu_0}{\sigma_0^2}, \qquad \alpha_0 = \mu_0 \beta_0 = \frac{\mu_0^2}{\sigma_0^2}.$$
   De exemplu, pentru $\mu_0 = 18$ apeluri/oră și $\sigma_0 = 6$ apeluri/oră, obținem $\beta_0 = 0.5, \alpha_0 = 9$, iar distribuția a posteriori devine $\text{Gamma}(189, 10.5)$ (cu modul $17.905$ și HDI 94% $[15.56, 20.48]$ — practic identic datorită volumului mare de date observate).

In [ ]:
import numpy as np
from scipy import stats, optimize
import matplotlib.pyplot as plt

# Datele observate
n_ore = 10
total_apeluri = 180

# Funcție pentru calculul exact al intervalului HDI (Highest Density Interval)
def calculeaza_hdi_gamma(alpha, beta, prob=0.94):
    dist = stats.gamma(a=alpha, scale=1.0 / beta)
    def latime_interval(coada_stanga):
        return dist.ppf(coada_stanga + prob) - dist.ppf(coada_stanga)
    rez = optimize.minimize_scalar(latime_interval, bounds=(0.0, 1.0 - prob), method="bounded")
    return dist.ppf(rez.x), dist.ppf(rez.x + prob)

# Comparăm priorul slab informativ cu un prior informativ istoric
scenarii = [
    ("Prior slab informativ (α₀=1, β₀=0.01)", 1.0, 0.01),
    ("Prior informativ istoric (μ₀=18, σ₀=6 ⇒ α₀=9, β₀=0.5)", 9.0, 0.5),
]

for nume, a0, b0 in scenarii:
    a_post = a0 + total_apeluri
    b_post = b0 + n_ore
    modul = max(a_post - 1.0, 0.0) / b_post
    media = a_post / b_post
    std = np.sqrt(a_post) / b_post
    hdi_low, hdi_high = calculeaza_hdi_gamma(a_post, b_post, prob=0.94)
    print(f"=== {nume} ===")
    print(f"Distribuția a posteriori: Gamma(α = {a_post:.2f}, β = {b_post:.2f})")
    print(f"Cea mai probabilă valoare (Modul MAP): {modul:.4f} apeluri/oră")
    print(f"Media a posteriori:                    {media:.4f} apeluri/oră (SD = {std:.4f})")
    print(f"Interval HDI 94%:                      [{hdi_low:.4f}, {hdi_high:.4f}]\n")

# Eșantionare din distribuția a posteriori (cazul principal: α₀=1, β₀=0.01) și vizualizare cu ArviZ
rng = np.random.default_rng(42)
alpha_post, beta_post = 1.0 + total_apeluri, 0.01 + n_ore
esantioane_post = rng.gamma(shape=alpha_post, scale=1.0 / beta_post, size=100_000)

try:
    import arviz as az
    az.plot_posterior(esantioane_post, hdi_prob=0.94, point_estimate="mode")
    plt.title(f"Distribuția a posteriori Gamma({alpha_post:.0f}, {beta_post:.2f}) — HDI 94% și Modul")
    plt.xlabel("Rata medie de apeluri pe oră (λ)")
    plt.tight_layout()
    plt.show()
except ImportError:
    # Fallback grafic identic stilistic cu arviz.plot_posterior dacă arviz nu este instalat
    hdi_l, hdi_h = calculeaza_hdi_gamma(alpha_post, beta_post, prob=0.94)
    modul_exact = (alpha_post - 1.0) / beta_post
    x = np.linspace(13, 24, 500)
    y = stats.gamma.pdf(x, a=alpha_post, scale=1.0 / beta_post)

    plt.figure(figsize=(9, 4.5))
    plt.plot(x, y, color="#1f77b4", lw=2.2, label=f"Densitate a posteriori Gamma({alpha_post:.0f}, {beta_post:.2f})")
    plt.axvline(modul_exact, color="#d62728", linestyle="--", lw=1.8, label=f"Modul (MAP) = {modul_exact:.2f}")
    plt.hlines(y=0.005, xmin=hdi_l, xmax=hdi_h, color="black", lw=4, label=f"94% HDI: [{hdi_l:.2f}, {hdi_h:.2f}]")
    plt.fill_between(x, y, where=(x >= hdi_l) & (x <= hdi_h), color="#1f77b4", alpha=0.25)
    plt.title("Distribuția a posteriori pentru λ (94% HDI și Modul)")
    plt.xlabel("Rata medie de apeluri pe oră (λ)")
    plt.ylabel("Densitate de probabilitate")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()